In [4]:
# Load environment variables from project root
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [5]:
# Create Databricks Spark session
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
# spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.gold')

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, DoubleType

# Silver feature tables (one row per package)
silver_github = spark.table('ddc_databricks.silver.github_package_features')
silver_pypi = spark.table('ddc_databricks.silver.pypi_package_features')
silver_so = spark.table('ddc_databricks.silver.so_package_features')

# Silver text tables for sentiment analysis
silver_so_questions = spark.table('ddc_databricks.silver.so_questions')
silver_so_answers = spark.table('ddc_databricks.silver.so_answers')
silver_readmes = spark.table('ddc_databricks.silver.github_readmes')
silver_pypi_metadata = spark.table('ddc_databricks.silver.pypi_metadata')

In [8]:
# Sentiment analysis UDF — tries VADER, then TextBlob, then keyword heuristic.

sentiment_schema = StructType([
    StructField('compound', DoubleType(), True),
    StructField('positive', DoubleType(), True),
    StructField('negative', DoubleType(), True),
    StructField('neutral', DoubleType(), True),
])

_SENTIMENT_MAX_CHARS = 5000

def _compute_sentiment(text):
    if text is None or len(str(text).strip()) == 0:
        return (0.0, 0.0, 0.0, 1.0)
    snippet = str(text)[:_SENTIMENT_MAX_CHARS]

    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        scores = SentimentIntensityAnalyzer().polarity_scores(snippet)
        return (float(scores['compound']), float(scores['pos']),
                float(scores['neg']), float(scores['neu']))
    except ImportError:
        print("Vader sentiment not available")
        pass

    try:
        from textblob import TextBlob
        p = TextBlob(snippet).sentiment.polarity
        return (float(p), float(max(0.0, p)),
                float(max(0.0, -p)), float(1.0 - abs(p)))
    except ImportError:
        print("TextBlob sentiment not available")
        pass

    lower = snippet.lower()
    _POS = ['good', 'great', 'excellent', 'easy', 'fast', 'useful', 'best',
            'perfect', 'works', 'love', 'helpful', 'nice', 'clean', 'awesome']
    _NEG = ['bad', 'error', 'bug', 'slow', 'broken', 'fail', 'crash',
            'wrong', 'issue', 'problem', 'terrible', 'awful', 'hate', 'worst']
    pos = sum(1 for w in _POS if w in lower)
    neg = sum(1 for w in _NEG if w in lower)
    total = pos + neg
    if total == 0:
        return (0.0, 0.0, 0.0, 1.0)
    compound = (pos - neg) / total
    return (float(compound), float(pos / total),
            float(neg / total), float(1.0 - abs(compound)))

sentiment_udf = F.udf(_compute_sentiment, sentiment_schema)